# Did the policy work, or did it leak?

Every design in this notebook answers a causal question about a policy applied
to part of a map. They differ in which assumption they are willing to drop.

The textbook difference-in-differences asks whether the treated units moved more
than the untreated ones. On a map that question has a hidden premise: that the
untreated units were *unaffected*. A policy with any spatial reach — a transport
link, a tax change, a factory closing — violates it, and the violation does not
merely add noise. It biases the headline estimate, because the comparison group
has been moved by the very thing being measured.

Three designs follow, in increasing order of how much they give up:

1. **Spatial DiD** keeps the whole sample and adds an exposure term, so the
   spillover is estimated rather than assumed away.
2. **Spatial synthetic control** keeps one treated unit and *discards* its
   neighbours from the donor pool, buying a clean counterfactual with data.
3. **Shift-share** abandons unit-level treatment entirely and identifies from
   the interaction of common shocks with regional exposure.

Every dataset here is simulated, and every generating equation is printed beside
the estimates. That is deliberate: it is the only way to say whether an
estimator recovered the answer, rather than merely that it produced one.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using LinearAlgebra, SparseArrays, Statistics, Printf, Random

## 1. A region, and a policy applied to part of it

A 14 x 14 rook lattice: 196 units, each neighbouring the four it shares an edge
with. The policy covers a 5 x 5 block in the interior, so the treated area has a
boundary, and units just outside that boundary are the ones a spillover would
reach first.

The lattice is not realism, it is control. Its neighbour structure is known
exactly, so when an estimate misses the truth there is no question of whether
the weight matrix was wrong.

In [ ]:
side   = 14
N      = side^2
W      = make_rook_weights(side)
coords = hcat(repeat(1:side, side), repeat(1:side, inner = side)) .* 1.0

D = zeros(N)
for i in 1:N
    r, c = coords[i, 1], coords[i, 2]
    (5 <= r <= 9 && 5 <= c <= 9) && (D[i] = 1.0)
end

@printf("units            : %d\n", N)
@printf("treated          : %d\n", Int(sum(D)))
@printf("untreated        : %d\n", Int(N - sum(D)))

## 2. Exposure, and whether it is identified

`exposure` turns the treatment vector into the share of each unit's neighbours
that are treated. Row-standardised weights make it a fraction, so an untreated
unit with two treated neighbours out of four has exposure 0.5.

The number worth looking at is not the mean exposure but its **variance among
the untreated**. The spillover coefficient is identified from exactly that
variation: if every control unit had the same exposure, the exposure column
would be collinear with the intercept and the spillover would be
indistinguishable from the time effect. `check_interference` reports this
before anything is estimated, which is the right order — a diagnostic that only
arrives with the standard errors arrives too late to change the design.

In [ ]:
em = exposure(D, W)
E  = em.E[:, 1]

@printf("mean exposure, treated   : %.4f\n", mean(E[D .== 1.0]))
@printf("mean exposure, untreated : %.4f\n", mean(E[D .== 0.0]))
@printf("variance among untreated : %.6f\n", var(E[D .== 0.0]))
@printf("untreated with any       : %d of %d\n",
        count(>(0.0), E[D .== 0.0]), count(==(0.0), D))

println()
println(check_interference(em))

## 3. A known truth, and what a naive DiD does to it

Two periods. Unit effects persist, a common time effect shifts everyone, the
policy adds `tau` to the treated, and exposure adds `delta` times exposure to
everyone who has any:

    y[i, 1] = mu[i]                          + noise
    y[i, 2] = mu[i] + gamma + tau*D[i] + delta*E[i] + noise

The spillover is **negative** here: `delta = -1.5`. A policy that pulls activity
out of surrounding areas — a new retail park, a relocated public office — has
this shape. The sign matters for the direction of the bias, which is the point
of choosing it rather than leaving it positive.

The naive two-by-two has no term for `E`, so the spillover has nowhere to go. It
lands in both halves of the subtraction: the controls nearest the block were
pushed down, which flatters the treated by comparison.

In [ ]:
gamma, tau, delta = 0.5, 3.0, -1.5
rng = MersenneTwister(2026)
mu  = 2.0 .* randn(rng, N)

y = zeros(N, 2)
y[:, 1] .= mu .+ 0.4 .* randn(rng, N)
y[:, 2] .= mu .+ gamma .+ tau .* D .+ delta .* E .+ 0.4 .* randn(rng, N)

dy    = y[:, 2] .- y[:, 1]
naive = mean(dy[D .== 1.0]) - mean(dy[D .== 0.0])

@printf("truth, direct effect tau : %+.4f\n", tau)
@printf("naive 2x2 DiD            : %+.4f\n", naive)
@printf("error                    : %+.4f  (%.1f%% of the truth)\n",
        naive - tau, 100 * (naive - tau) / tau)

## 4. The spillover design

`fit_spatial_did` puts the exposure column in the regression and reports four
numbers instead of one:

| | what it is |
|---|---|
| `att_direct` | the effect on a treated unit, net of what reached it through its neighbours |
| `att_spill_treated` | the part of the treated outcome that arrived via exposure |
| `att_spill_control` | the same coefficient, scaled by the controls' mean exposure |
| `att_total_treated` | direct plus spillover, the total change a treated unit saw |

The two spillover figures differ because the groups have different average
exposure, **not** because two coefficients were estimated. There is one exposure
coefficient. This matters when reading the literature: Chagas, Azzoni and
Almeida (2016) estimate a genuinely two-channel version with separate
coefficients, and that is a different estimator which this function does not
fit. The design here is Delgado and Florax (2015).

In [ ]:
m = fit_spatial_did(y, D, W)

E_bar_T = mean(E[D .== 1.0])
E_bar_C = mean(E[D .== 0.0])

@printf("%-26s %10s %10s %9s %8s\n", "", "estimate", "truth", "error", "in SEs")
for (lab, est, want, se) in (
        ("direct effect",          m.att_direct,        tau,             m.att_direct_se),
        ("spillover, on treated",  m.att_spill_treated, delta * E_bar_T, m.att_spill_treated_se),
        ("spillover, on controls", m.att_spill_control, delta * E_bar_C, m.att_spill_control_se))
    @printf("%-26s %10.4f %10.4f %+9.4f %8.2f\n",
            lab, est, want, est - want, abs(est - want) / se)
end

println()
@printf("the naive estimate was off by  %+.4f\n", naive - tau)
@printf("the spillover design, off by   %+.4f\n", m.att_direct - tau)

## 5. When the control group is the problem: spatial synthetic control

Spatial DiD estimates the contamination. Synthetic control takes the other
route: it refuses to use contaminated units at all.

One treated region, many candidate donors, and a weighted average of donors
built to track the treated unit's pre-treatment path. The conventional
implementation draws donors from every untreated unit — including the treated
unit's immediate neighbours, which are the units most likely to have absorbed a
spillover. Weight placed on them puts part of the treatment effect into the
counterfactual, and the estimate shrinks towards zero.

`donor_pool = :exclude_neighbours` is the default here, and the comparison below
shows what the default buys. Note the trade: excluding neighbours throws away
the donors that usually fit best, so the pre-treatment RMSPE is *worse*. A
better pre-fit is not the goal; an uncontaminated one is.

In [ ]:
side_s   = 9
N_s      = side_s^2
W_s      = make_rook_weights(side_s)
coords_s = hcat(repeat(1:side_s, side_s), repeat(1:side_s, inner = side_s)) .* 1.0

rng2  = MersenneTwister(2026)
T, T0 = 20, 12
nf    = 3
F     = cumsum(randn(rng2, T, nf), dims = 1)     # three common factors

# Each unit's loading on each factor is a spatially autocorrelated field, so a
# unit's pre-treatment path is close to its neighbours' and far from everyone
# else's. Matching on three factors at once is what makes proximity, rather
# than coincidence, the reason a donor fits.
smooth(r) = (I - 0.9 .* Matrix(W_s)) \ randn(r, N_s)
L = hcat([let v = smooth(rng2); 1.0 .+ 0.6 .* (v ./ std(v)) end for _ in 1:nf]...)

Y = L * F' .+ 0.03 .* randn(rng2, N_s, T)

treated        = 41                              # the centre, (5,5)
att_true       = 3.0
spillover_true = 2.0

Ds = zeros(N_s); Ds[treated] = 1.0
Es = W_s * Ds                                    # neighbours get 1/4 each

for t in (T0 + 1):T
    Y[treated, t] += att_true
    Y[:, t]      .+= spillover_true .* Es
end

scm_all = fit_synthetic_control(Y, treated, T0; donor_pool = :all, penalty_metric = :none)
scm     = fit_synthetic_control(Y, treated, T0; W = W_s, coords = coords_s)

contaminated = sum(scm_all.weights[1, q]
                   for (q, j) in enumerate(scm_all.donors) if Es[j] > 0.0; init = 0.0)

@printf("%-34s %10s %10s %12s\n", "", "att", "error", "pre-RMSPE")
@printf("%-34s %10.4f %+10.4f %12.5f\n", "all donors (conventional)",
        scm_all.att, scm_all.att - att_true, scm_all.rmspe_pre[1])
@printf("%-34s %10.4f %+10.4f %12.5f\n", "neighbours excluded (default)",
        scm.att, scm.att - att_true, scm.rmspe_pre[1])
@printf("%-34s %10.4f\n", "truth", att_true)

println()
@printf("weight the naive pool put on exposed donors : %.4f\n", contaminated)
@printf("donors available after exclusion            : %d of %d\n",
        length(scm.donors), N_s - 1)

println("\nwhere the naive pool put its weight:")
ord = sortperm(vec(scm_all.weights[1, :]), rev = true)
for q in ord[1:5]
    j = scm_all.donors[q]
    @printf("  unit %3d  weight %.4f  exposure %.2f %s\n",
            j, scm_all.weights[1, q], Es[j], Es[j] > 0 ? "<- neighbour of the treated" : "")
end

Read the two rows against each other. The conventional pool fits the
pre-treatment path *better* — a lower RMSPE — and gets the answer *worse*,
because around half its weight sits on a single immediate neighbour whose
post-treatment outcome contains the spillover. The counterfactual is therefore
built partly out of the treatment effect, and the att is pulled towards zero.
Neighbour exclusion gives up that donor, fits less well, and lands closer.

A better pre-fit is not the objective. An uncontaminated one is.

### Is one estimate enough?

With a single treated unit there is no sampling distribution to appeal to, so
inference is permutation-based: refit the whole procedure pretending each donor
was treated, and ask where the real estimate falls among the placebos.

`placebo_test` reports that, and it also reports what it *cannot* do. The
smallest attainable p-value is bounded below by `1 / (n_donors + 1)`, so with
76 donors nothing below about 0.013 is reachable however large the effect.
Here the observed p-value **is** that floor, which is the most the design can
say rather than a coincidence worth remarking on. `min_attainable_p` makes the
bound explicit instead of leaving a reader to infer a precision the permutation
cannot deliver.

In [ ]:
pt = placebo_test(scm, Y; n_placebo = 200, rng = MersenneTwister(7))

@printf("observed post/pre RMSPE ratio : %.4f\n", pt.ratio_observed)
@printf("placebo draws                 : %d\n", pt.n_placebo)
@printf("p-value                       : %.4f\n", pt.p_value)
@printf("smallest p this design allows  : %.4f\n", pt.min_attainable_p)

## 6. Shift-share: no treated unit at all

The third design drops unit-level treatment. Each region holds a bundle of
industries; a shock hits each industry nationally; a region's exposure is its
industry mix. The regressor is the inner product of shares and shocks, and
identification comes from the shocks being plausibly exogenous to the regions.

The reason this needs its own standard error is **exposure concentration**. If
one industry dominates the shares, the effective number of independent shocks is
far smaller than the number of regions, and conventional heteroskedasticity-
robust errors are computed as though each region were an independent draw. The
AKM standard error of Adao, Kolesar and Morales (2019) accounts for the shock
structure instead.

`exposure_concentration` reports a Herfindahl index over average exposure shares
and converts it to an effective shock count, which is the number to compare
against the region count before trusting an HC1 error.

In [ ]:
RNG  = MersenneTwister(11)
N_ss = 300
K    = 8
BETA = 1.0

# Concentrated exposure: a few industries carry most of the weight.
S = rand(RNG, N_ss, K) .^ 3
S[:, 1] .*= 12.0
S ./= sum(S, dims = 2)

g = randn(RNG, K)                               # the industry shocks

W_ss = S * S'                                   # share-overlap adjacency
W_ss[diagind(W_ss)] .= 0.0
W_ss = sparse(W_ss ./ max.(sum(W_ss, dims = 2), eps()))

design = shift_share_design(S, g)
eta  = randn(RNG, K) .* 3.0
eps  = S * eta .+ 0.5 .* randn(RNG, N_ss)       # errors share the shock structure
y_ss = design.X .* BETA .+ eps

conc = exposure_concentration(design)
@printf("shocks                  : %d\n", conc.n_shocks)
@printf("exposure HHI            : %.4f\n", conc.hhi)
@printf("effective shocks        : %.2f\n", conc.effective_shocks)
@printf("largest single share    : %.4f\n", conc.max_share)
println()

f_akm = fit_shift_share(y_ss, design; vcov = :akm)
f_hc1 = fit_shift_share(y_ss, design; vcov = :hc1)

@printf("beta (truth %.1f)        : %.4f\n", BETA, f_akm.beta)
@printf("AKM standard error      : %.4f\n", f_akm.se)
@printf("HC1 standard error      : %.4f\n", f_hc1.se)
@printf("AKM / HC1               : %.2fx\n", f_akm.se / f_hc1.se)

## 7. What to carry away

- A spillover is not noise. With `delta = -1.5` the naive two-by-two missed the
  direct effect by roughly a third of its own size, in a predictable direction:
  negative spillovers depress the comparison group and so flatter the treated.
  The exposure design cuts that error substantially, though it does not abolish
  it — read the "in SEs" column rather than the point estimates.
- The exposure coefficient is identified by the **variance of exposure among
  the untreated**. Check it before estimating; `check_interference` reports it
  in advance, and a design where every control is equally exposed cannot
  separate spillover from time effect at all.
- Synthetic control's neighbour exclusion costs pre-treatment fit and buys an
  uncontaminated counterfactual. Here it fit worse and was six times closer to
  the truth. Judge the donor pool on the att, never on the RMSPE.
- With one treated unit, inference is permutational and floored at
  `1 / (n_donors + 1)`. Read `min_attainable_p` before reading the p-value.
- Under concentrated exposure the AKM standard error was about five times the
  HC1 one on the same coefficient. With 8 shocks and roughly 4 effective ones
  behind 300 regions, HC1 is counting independent observations that are not
  there.

The limitation shared by all three designs: every dataset here was generated
from the model being fitted. That is what makes a comparison against truth
possible, and it is also why none of these numbers is evidence about behaviour
under misspecification. `examples/13_simulation_power.jl` and the Monte Carlo
studies in `validation/` are where that is examined.